# 33 — ShipCrew: Multi-Agent Orchestration

OpenClaw-style multi-agent crews where specialized agents collaborate on tasks autonomously. DAG-based task dependencies, parallel execution, and template variable resolution.

**What you'll learn:**
1. Creating a basic crew with two agents
2. Task dependencies and data flow
3. Sequential, parallel, and hierarchical modes
4. Context variables
5. Streaming crew events
6. Loading agents from the registry
7. The `create_ship_crew` factory
8. Validation and error handling

In [1]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent
from shipit_agent.deep.ship_crew import (
    ShipCrew,
    ShipAgent,
    ShipTask,
    create_ship_crew,
)
from shipit_agent.deep.ship_crew.errors import (
    ShipCrewError,
    CyclicDependencyError,
)
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env("bedrock")

## 1. Your First ShipCrew

A crew has **agents** (who do the work) and **tasks** (what to do). Each task is assigned to an agent. Tasks can depend on each other — output flows via `{output_key}` templates.

In [2]:
# Create two specialized agents
researcher = ShipAgent(
    name="researcher",
    agent=Agent(
        llm=llm,
        prompt="You are a thorough researcher. Provide detailed, factual findings.",
    ),
    role="Senior Researcher",
    goal="Find comprehensive, accurate information",
    capabilities=["research", "analysis", "summarization"],
)

writer = ShipAgent(
    name="writer",
    agent=Agent(
        llm=llm,
        prompt="You are a skilled technical writer. Write clear, engaging content.",
    ),
    role="Technical Writer",
    goal="Transform research into polished content",
    capabilities=["writing", "editing", "formatting"],
)

reviewer = ShipAgent(
    name="reviewer",
    agent=Agent(
        llm=llm,
        prompt="You are a meticulous reviewer. Provide constructive feedback and ensure quality.",
    ),
    role="Content Reviewer",
    goal="Review content for accuracy and clarity",
    capabilities=["critical analysis", "feedback", "quality assurance"],
)

# Define tasks with dependencies
crew = ShipCrew(
    name="content-crew",
    coordinator_llm=llm,
    agents=[researcher, writer, reviewer],
    tasks=[
        ShipTask(
            name="research",
            description="Research the current state of AI agents in 2026. Cover key frameworks, trends, and adoption.",
            agent="researcher",
            output_key="findings",
        ),
        ShipTask(
            name="write",
            description="Write a concise 3-paragraph summary based on these findings: {findings}",
            agent="writer",
            depends_on=["research"],
            output_key="article",
        ),
        ShipTask(
            name="review",
            description="Review the article for accuracy and clarity. Provide feedback and suggest improvements.",
            agent="reviewer",
            depends_on=["write"],
            output_key="reviewed_article",
        ),
    ],
)

In [6]:
result = crew.run()
print(f"Execution order: {result.execution_order}")
print(f"Total tasks:     {result.total_tasks}")
print(f"Failed tasks:    {result.failed_tasks}")
print(f"Time:            {result.metadata.get('elapsed_seconds', 'n/a')}s")

print("\n=== Research Findings (preview) ===")
print(result.task_results.get("findings", "")[:400])

print("\n=== Final Article ===")

Execution order: ['analyze', 'research', 'draft', 'review']
Total tasks:     4
Failed tasks:    []
Time:            90.91s

=== Research Findings (preview) ===


=== Final Article ===


In [7]:
display(Markdown(result.output))

**Content Review – Draft Report Outline**

Overall, the draft is clear, well‑structured, and covers the essential components of a market‑analysis report. Below are a few observations and suggestions to improve accuracy, consistency, and readability.

---

### 1. Table Formatting & Consistency
| Issue | Recommendation |
|-------|----------------|
| **Section 8 title** – “visualizations” is lowercase while all other headings are title‑cased. | Change to **“8. Visualizations”** for visual parity. |
| **Section 8 description** – “Charts/graphs (trend lines, bar charts, heat maps) that make the insights instantly clear.” <br>Consider adding “and any other custom visual types the client requests” to keep it open‑ended. | Add “…or any other custom visual types the client requests.” |
| **Section 10 – Appendices** – “None needed beyond the dataset itself.” <br>In practice you may still need a data‑dictionary or methodology note even if the client only supplies the raw data. | Re‑phrase to: “Append any supplementary material (e.g., data‑dictionary, methodology notes) that supports the analysis.” |
| **Column header “Data I’ll need from you”** – The phrasing is conversational but can be tightened for a formal report outline. | Consider “Data Required from Client” or keep the conversational tone but ensure parallelism across rows. |

---

### 2. Clarity of Requests
- **Executive Summary** – You ask for a “Target time frame”. It may be helpful to also request “primary business objectives” (e.g., market entry, pricing strategy) here, as they shape the executive summary.
- **Research Context** – The placeholder text “A short narrative or link describing the market you’re studying” is fine, but you could ask for “any existing market research or analyst reports” that should be referenced.
- **Dataset Overview** – Ask for “data‑source provenance” (who collected the data, date of collection) to flag potential bias or staleness early.
- **Descriptive Statistics** – You request the “most important quantitative metrics”. Adding a note to specify “units of measurement” (e.g., USD, units sold) will prevent mis‑interpretation.
- **Trend & Time‑Series Analysis** – It’s useful to ask if the client expects any “forecasting” beyond descriptive trends (e.g., ARIMA, exponential smoothing). If not, state that the analysis is descriptive only.
- **Segmentation & Comparative Insights** – Clarify whether the client wants “cross‑tabulations” (e.g., region × product) or simple single‑dimensional splits.
- **KPIs** – Good to ask whether the client needs “derived metrics” (e.g., LTV, CAC) that may require additional assumptions.
- **Recommendations** – In addition to business goals, ask for any “constraints” (budget, regulatory, resource limits) that could shape actionable advice.

---

### 3. Minor Typographical / Stylistic Points
- **Section 8 heading**: should be capitalised (“Visualizations”).
- **Em dash usage** – In the introductory paragraph, “Below is a quick draft of the report structure I’d use once I have the market data you’d like analysed.” The phrase “you’d like analysed” could be smoother as “you’d like analyzed” (US spelling) or “you’d like analysed” (UK). Choose one style and keep it consistent throughout the document.
- **Bullet list under “📦 What I need from you right now”** – The bullet “The dataset – CSV, Excel, Google Sheet link, or a small sample (≈ 20–30 rows will let me verify the schema).” could be split into two bullets (format vs. sample size) for easier scanning.
- **Use of non‑breaking spaces** – In the table (e.g., “FY 2023”) you have narrow spaces; ensure they render correctly across platforms, or replace with regular spaces.

---

### 4. Completeness Check
- **Assumptions & Limitations** – Consider adding a dedicated “Assumptions & Limitations” subsection (perhaps under “Research Context” or as its own section) so the client knows what caveats accompany the analysis.
- **Validation & Quality Assurance** – It may be useful to note that you will perform data‑quality checks (duplicate detection, outlier handling) before analysis. This reassures the client that the numbers are trustworthy.
- **Deliverable Format** – Clarify whether the final report will be a PDF, PowerPoint deck, interactive dashboard, or a combination. This helps set expectations early.

---

### 5. Suggested Revised Outline (lightly edited)

| Section | What I’ll cover | Data Required from Client |
|---------|----------------|---------------------------|
| **1. Executive Summary** | One‑page snapshot of key findings (market size, growth rate, top opportunities). | • Target time frame (e.g., Q1‑2024, FY 2023). <br>• Primary business objectives (growth, entry, cost‑reduction). |
| **2. Research Context** | Market description, data sources, assumptions, and any existing reports. | • Brief narrative or link describing the market. <br>• Any prior research or analyst reports. |
| **3. Dataset Overview** | Variable list, record count, provenance, data‑quality notes (missing values, outliers). | • Raw dataset (CSV, Excel, Google Sheet, or sample). <br>• Source/collection date and ownership. |
| **4. Descriptive Statistics** | Central tendency & dispersion for each quantitative metric; frequency tables for categorical fields. | • List of quantitative metrics to prioritize (revenue, unit sales, price, etc.). |
| **5. Trend & Time‑Series Analysis** | Growth/decline patterns, seasonal effects, optional forecasting. | • Date/period column name and granularity (daily, weekly, monthly, quarterly). |
| **6. Segmentation & Comparative Insights** | Breakdowns by region, product line, customer segment; top‑ vs. under‑performing groups. | • Desired segmentation dimensions (region, product category, customer size, etc.). |
| **7. Key Performance Indicators (KPIs)** | Market share, CAGR, churn rate, average deal size, LTV/CAC (if applicable). | • Specific KPIs to calculate (or allow suggestions). |
| **8. Visualizations** | Charts/graphs (trend lines, bar charts, heat maps, custom visuals) to illustrate insights. | • Visual style preferences (color palette, branding, logo). |
| **9. Recommendations** | Actionable take‑aways aligned with business goals and any known constraints. | • Business goals and constraints (budget, regulatory, resource limits). |
| **10. Appendices** | Full tables, methodology notes, data‑dictionary, assumptions & limitations. | • None beyond the dataset (unless additional supporting docs are needed). |

---

### 6. Final Thoughts
- The outline is comprehensive and already client‑friendly.  
- Minor wording/format tweaks (as above) will enhance professionalism and reduce ambiguity.  
- Adding explicit sections for **Assumptions & Limitations** and **Quality Assurance** will strengthen credibility.  
- Clarifying the **final deliverable format** helps manage expectations.

Feel free to incorporate any of these suggestions or let me know if you need a more detailed rewrite of a specific section.

## 2. Diamond DAG — Multiple Dependencies

Tasks form a directed acyclic graph. Independent tasks can run in parallel; dependent tasks wait.

```
research ──┐
           ├──→ draft ──→ review
analyze ───┘
```

In [4]:
analyst = ShipAgent(
    name="analyst",
    agent=Agent(
        llm=llm, prompt="You are a data analyst. Provide concise numerical insights."
    ),
    role="Data Analyst",
    goal="Extract data-driven insights",
)

reviewer = ShipAgent(
    name="reviewer",
    agent=Agent(
        llm=llm, prompt="You are a critical reviewer. Verify facts and flag issues."
    ),
    role="Content Reviewer",
    goal="Ensure accuracy and quality",
)

crew = ShipCrew(
    name="report-crew",
    coordinator_llm=llm,
    agents=[researcher, analyst, writer, reviewer],
    tasks=[
        ShipTask(
            name="research",
            description="Research AI agent market size and growth",
            agent="researcher",
            output_key="research",
        ),
        ShipTask(
            name="analyze",
            description="Analyze market data and provide key statistics",
            agent="analyst",
            output_key="analysis",
        ),
        ShipTask(
            name="draft",
            description="Write a report combining:\nResearch: {research}\nAnalysis: {analysis}",
            agent="writer",
            depends_on=["research", "analyze"],
            output_key="draft",
        ),
        ShipTask(
            name="review",
            description="Review this draft for accuracy: {draft}",
            agent="reviewer",
            depends_on=["draft"],
        ),
    ],
)

errors = crew.validate()
print("Valid!" if not errors else f"Errors: {errors}")

result = crew.run()
print(f"Execution order: {result.execution_order}")
print(f"Time: {result.metadata.get('elapsed_seconds', 'n/a')}s")
display(Markdown(result.output[:600]))

Valid!


open_url: Playwright fetch failed for https://www.statista.com/statistics/1552183/global-agentic-ai-market-value/: HTTP 401 from https://www.statista.com/statistics/1552183/global-agentic-ai-market-value/
open_url: urllib fetch failed for https://www.statista.com/statistics/1552183/global-agentic-ai-market-value/: HTTP Error 401: Unauthorized


KeyboardInterrupt: 

## 3. Parallel Execution Mode

In `process="parallel"`, independent tasks in the same DAG layer run concurrently via `ThreadPoolExecutor`.

In [ ]:
crew_parallel = ShipCrew(
    name="parallel-crew",
    coordinator_llm=llm,
    agents=[researcher, analyst, writer],
    tasks=[
        ShipTask(
            name="research",
            description="Research AI agent frameworks",
            agent="researcher",
            output_key="research",
        ),
        ShipTask(
            name="analyze",
            description="Analyze AI adoption trends",
            agent="analyst",
            output_key="analysis",
        ),
        ShipTask(
            name="draft",
            description="Combine:\n{research}\n{analysis}",
            agent="writer",
            depends_on=["research", "analyze"],
        ),
    ],
    process="parallel",
)

result = crew_parallel.run()
print(f"Execution order: {result.execution_order}")
print(f"Time: {result.metadata.get('elapsed_seconds', 'n/a')}s")
print("'research' and 'analyze' ran in parallel (same DAG layer)")
display(Markdown(result.output[:500]))

## 3b. Context Variables

Pass runtime variables into your crew. `{topic}` and `{audience}` in task descriptions resolve from `crew.run(topic=..., audience=...)`.

In [ ]:
crew_ctx = ShipCrew(
    name="flexible-crew",
    coordinator_llm=llm,
    agents=[researcher, writer],
    tasks=[
        ShipTask(
            name="research",
            description="Research {topic} for a {audience} audience",
            agent="researcher",
            output_key="findings",
        ),
        ShipTask(
            name="write",
            description="Write a {format} about {topic} based on: {findings}",
            agent="writer",
            depends_on=["research"],
        ),
    ],
)

result = crew_ctx.run(
    topic="quantum computing breakthroughs",
    audience="executive leadership",
    format="2-paragraph executive brief",
)
display(Markdown(result.output[:600]))

## 4. Hierarchical (LLM-Driven) Mode

In `process="hierarchical"`, the coordinator LLM dynamically decides which task to assign next, to which agent, and can request revisions.

In [ ]:
crew_hierarchical = ShipCrew(
    name="hierarchical-demo",
    coordinator_llm=llm,
    agents=[researcher, analyst, writer],
    tasks=[
        ShipTask(
            name="research",
            description="Research AI agent safety concerns",
            agent="researcher",
            output_key="research",
        ),
        ShipTask(
            name="analyze",
            description="Analyze the severity of each concern",
            agent="analyst",
            output_key="analysis",
        ),
        ShipTask(
            name="report",
            description="Write a safety report from {research} and {analysis}",
            agent="writer",
            depends_on=["research", "analyze"],
        ),
    ],
    process="hierarchical",
    max_rounds=8,
)

result = crew_hierarchical.run()
print("Mode: hierarchical")
print(f"Execution order: {result.execution_order}")
print(
    f"Tasks completed: {result.total_tasks - len(result.failed_tasks)}/{result.total_tasks}"
)
print(f"Time: {result.metadata.get('elapsed_seconds', 'n/a')}s")
display(Markdown(result.output[:600]))

## 5. Streaming Crew Events

Monitor execution in real-time with `crew.stream()`.

In [9]:
crew_stream = ShipCrew(
    name="stream-demo",
    coordinator_llm=llm,
    agents=[researcher, writer],
    tasks=[
        ShipTask(
            name="research",
            description="Research top 3 AI trends for 2026",
            agent="researcher",
            output_key="findings",
        ),
        ShipTask(
            name="write",
            description="Summarize in 2 sentences: {findings}",
            agent="writer",
            depends_on=["research"],
        ),
        ShipTask(
            name="review",
            description="Expand the summary into a 3-paragraph article",
            agent="writer",
            depends_on=["write"],
        ),
        ShipTask(
            name="finalize",
            description="Polish the article and ensure it's well-formatted",
            agent="writer",
            depends_on=["review"],
        ),
        ShipTask(
            name="publish",
            description="Publish the article to the company blog",
            agent="writer",
            depends_on=["finalize"],
        ),
        ShipTask(
            name="bonus",
            description="Write a catchy headline for the article",
            agent="writer",
            depends_on=["write"],
        ),
    ],
)

print("=== Streaming Crew Events ===\n")
for event in crew_stream.stream():
    emoji = {
        "run_started": "🚀",
        "tool_called": "⚙️",
        "tool_completed": "✅",
        "tool_failed": "❌",
        "run_completed": "🏁",
    }.get(event.type, "📌")
    print(f"{emoji} [{event.type:16s}] {event.message[:100]}")
    if event.type == "run_completed":
        print("\n=== Final Output ===")
        display(Markdown(event.payload.get("output", "")))

=== Streaming Crew Events ===

🚀 [run_started     ] Crew started (sequential mode, 6 tasks)
⚙️ [tool_called     ] Task 'research' started (agent: researcher)
✅ [tool_completed  ] Task 'research' completed
⚙️ [tool_called     ] Task 'write' started (agent: writer)
✅ [tool_completed  ] Task 'write' completed
⚙️ [tool_called     ] Task 'bonus' started (agent: writer)
✅ [tool_completed  ] Task 'bonus' completed
⚙️ [tool_called     ] Task 'review' started (agent: writer)
✅ [tool_completed  ] Task 'review' completed
⚙️ [tool_called     ] Task 'finalize' started (agent: writer)
✅ [tool_completed  ] Task 'finalize' completed
⚙️ [tool_called     ] Task 'publish' started (agent: writer)
✅ [tool_completed  ] Task 'publish' completed
🏁 [run_completed   ] Crew completed

=== Final Output ===


Absolutely—let’s make sure we have a crystal‑clear picture before we start crafting the final blog post.

### 1. Clarify the target output & inputs
Please let me know the details for each of the items below:

| Item | What I need from you |
|------|----------------------|
| **Article topic / title** | The main subject (e.g., “How AI is Revolutionizing Technical SEO”). |
| **Key research/material** | Links, PDFs, notes, or bullet points you’ve gathered that should be incorporated. |
| **Target au

In [ ]:
# Summarize the stream event mix for easier debugging
from collections import Counter

event_types = [event.type for event in crew_stream.stream()]
counts = Counter(event_types)
print("Event counts:")
for event_type, count in sorted(counts.items()):
    print(f"  {event_type:16s} {count}")

## 6. Loading Agents from the Registry

Use `ShipAgent.from_registry()` to build crew agents from prebuilt definitions.

In [ ]:
security_agent = ShipAgent.from_registry("security-auditor", llm=llm)
code_reviewer_agent = ShipAgent.from_registry("code-reviewer", llm=llm)

print(f"Security Agent: {security_agent.role}")
print(f"Code Reviewer:  {code_reviewer_agent.role}")

# Security review crew
security_crew = ShipCrew(
    name="security-review",
    coordinator_llm=llm,
    agents=[security_agent, code_reviewer_agent],
    tasks=[
        ShipTask(
            name="audit",
            description="Audit this code for vulnerabilities:\n```python\nimport os\ndef run_cmd(user_input):\n    os.system(f'echo {user_input}')\n```",
            agent=security_agent.name,
            output_key="audit_findings",
        ),
        ShipTask(
            name="review",
            description="Review audit and suggest fixes: {audit_findings}",
            agent=code_reviewer_agent.name,
            depends_on=["audit"],
        ),
    ],
)

result = security_crew.run()
display(Markdown(result.output[:800]))

## 7. The `create_ship_crew` Factory

Build crews from plain dicts — useful when loading from JSON config.

In [ ]:
crew = create_ship_crew(
    coordinator_llm=llm,
    agents=[
        {
            "name": "r",
            "agent": Agent(llm=llm, prompt="You research topics."),
            "role": "Researcher",
        },
        {
            "name": "w",
            "agent": Agent(llm=llm, prompt="You write summaries."),
            "role": "Writer",
        },
    ],
    tasks=[
        {
            "name": "research",
            "description": "Research cloud computing trends",
            "agent": "r",
            "output_key": "data",
        },
        {
            "name": "write",
            "description": "Summarize: {data}",
            "agent": "w",
            "depends_on": ["research"],
        },
    ],
    name="dict-crew",
)

result = crew.run()
print(f"Factory crew result: {result.output[:300]}")

## 8. Validation and Error Handling

Catch configuration errors before running.

In [ ]:
# Missing agent
bad_crew = ShipCrew(
    name="bad-crew",
    coordinator_llm=llm,
    agents=[researcher],
    tasks=[ShipTask(name="edit", description="Edit text", agent="editor")],
)
errors = bad_crew.validate()
print("Missing agent errors:")
for e in errors:
    print(f"  ❌ {e}")

# Cyclic dependency
try:
    ShipCrew(
        name="cycle",
        coordinator_llm=llm,
        agents=[researcher, writer],
        tasks=[
            ShipTask(
                name="a", description="A needs B", agent="researcher", depends_on=["b"]
            ),
            ShipTask(
                name="b", description="B needs A", agent="writer", depends_on=["a"]
            ),
        ],
    )
except CyclicDependencyError as e:
    print(f"\nCyclic dependency: {e}")

# Unknown dependency
try:
    ShipCrew(
        name="bad-dep",
        coordinator_llm=llm,
        agents=[researcher],
        tasks=[
            ShipTask(
                name="a",
                description="A",
                agent="researcher",
                depends_on=["nonexistent"],
            )
        ],
    )
except ShipCrewError as e:
    print(f"Unknown dep: {e}")

## 9. ShipTask Advanced Features

Tasks support retries, timeouts, extra context, and output schemas.

In [ ]:
# Task with retries, timeout, extra context, and an output schema
detailed_task = ShipTask(
    name="deep-research",
    description="Research {topic} with focus on {focus_area}",
    agent="researcher",
    depends_on=[],
    output_key="deep_findings",
    output_schema={"type": "object", "required": ["summary", "risks"]},
    max_retries=3,  # retry up to 3 times on failure
    timeout_seconds=120,  # 2 minute timeout
    context={"priority": "high", "depth": "comprehensive"},
)

print(f"Task: {detailed_task.name}")
print(f"  agent:       {detailed_task.agent}")
print(f"  output_key:  {detailed_task.output_key}")
print(f"  schema:      {detailed_task.output_schema}")
print(f"  max_retries: {detailed_task.max_retries}")
print(f"  timeout:     {detailed_task.timeout_seconds}s")
print(f"  context:     {detailed_task.context}")
print(f"  depends_on:  {detailed_task.depends_on}")

# Template resolution demo
resolved = detailed_task.resolve_description(
    {
        "topic": "neural architecture search",
        "focus_area": "efficiency gains",
    }
)
print(f"\nResolved description: {resolved}")

# Safe resolution — missing vars stay as {var}
partial = detailed_task.resolve_description({"topic": "NAS"})
print(f"Partial resolution:  {partial}")

# Serialization
import json

print(f"\nSerialized:\n{json.dumps(detailed_task.to_dict(), indent=2)}")

# Round-trip
restored = ShipTask.from_dict(detailed_task.to_dict())
print(f"\nRound-trip: name={restored.name}, retries={restored.max_retries}")

## 10. Combining ShipCrew with Cost Tracking

Wire a CostTracker into crew agents to monitor total spend across all agents.

In [ ]:
from shipit_agent.costs import CostTracker, Budget

# Create a cost tracker
tracker = CostTracker(budget=Budget(max_dollars=3.00))

# Build agents with cost-tracking hooks
tracked_researcher = ShipAgent(
    name="researcher",
    agent=Agent(llm=llm, prompt="You research topics.", hooks=tracker.as_hooks()),
    role="Researcher",
)
tracked_writer = ShipAgent(
    name="writer",
    agent=Agent(llm=llm, prompt="You write summaries.", hooks=tracker.as_hooks()),
    role="Writer",
)

cost_crew = ShipCrew(
    name="cost-tracked-crew",
    coordinator_llm=llm,
    agents=[tracked_researcher, tracked_writer],
    tasks=[
        ShipTask(
            name="research",
            description="Research microservices vs monolith",
            agent="researcher",
            output_key="findings",
        ),
        ShipTask(
            name="write",
            description="Write a comparison: {findings}",
            agent="writer",
            depends_on=["research"],
        ),
    ],
)

result = cost_crew.run()

print("=== Crew + Cost Tracking ===")
print(f"Total cost:   ${tracker.total_cost:.4f}")
print(f"Total calls:  {len(tracker.breakdown())}")
print(f"Total tokens: {tracker.total_tokens}")
print("\nPer-call:")
for c in tracker.breakdown():
    print(f"  #{c['call_number']}: {c['model'][:30]:30s} ${c['cost_usd']:.4f}")
display(Markdown(result.output[:400]))

## Summary

| Feature | API |
|---------|-----|
| Create a crew | `ShipCrew(name=..., coordinator_llm=llm, agents=[...], tasks=[...])` |
| Define a task | `ShipTask(name=..., description=..., agent=..., depends_on=[...])` |
| Wrap an agent | `ShipAgent(name=..., agent=..., role=..., goal=...)` |
| From registry | `ShipAgent.from_registry("security-auditor", llm=llm)` |
| Sequential mode | `process="sequential"` (default) |
| Parallel mode | `process="parallel"` |
| Hierarchical mode | `process="hierarchical"` |
| Context variables | `crew.run(topic="AI", audience="devs")` |
| Stream events | `for event in crew.stream(): ...` |
| Factory from dicts | `create_ship_crew(coordinator_llm=llm, agents=[...], tasks=[...])` |
| Validate config | `crew.validate()` → list of errors |

**Three execution modes. DAG-based dependencies. Template variable resolution. Production-ready.**